# 04 - Predicciones y Proyeccion de Costos
## Prueba Tecnica - Cientifico de Datos Senior

---

**Objetivo:** Cargar los modelos entrenados, proyectar el costo de los equipos para los meses futuros con intervalos de confianza, y justificar el horizonte de prediccion.

**Modelos disponibles:**
- Lasso (mejor modelo por metricas)
- Prophet (recomendado para proyeccion por intervalos de confianza nativos)

**Datos disponibles para proyeccion:**
- X.csv tiene datos hasta abril 2024 (154 dias despues del historico)
- Y.csv tiene datos hasta septiembre 2023 (8 dias despues)
- Z.csv termina igual que el historico (agosto 2023)
- Historico de equipos termina en agosto 2023

**Estrategia:** Usar Prophet para proyectar las materias primas Y y Z al futuro, luego usar esos valores proyectados junto con los datos reales de X para estimar los costos de los equipos.

## 0. Configuracion

In [ ]:
!pip install prophet --quiet

In [ ]:
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle
from io import StringIO
from prophet import Prophet
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_style('whitegrid')

BUCKET = 'consulting-dataknow-prueba-tecnica'
s3 = boto3.client('s3')

def leer_csv_s3(bucket, key, **kwargs):
    obj = s3.get_object(Bucket=bucket, Key=key)
    contenido = obj['Body'].read().decode('utf-8-sig')
    return pd.read_csv(StringIO(contenido), **kwargs)

def cargar_pickle_s3(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pickle.loads(obj['Body'].read())

def guardar_json_s3(data, bucket, key):
    s3.put_object(Bucket=bucket, Key=key, Body=json.dumps(data, default=str, ensure_ascii=False))
    print(f'Guardado en s3://{bucket}/{key}')

print('Entorno configurado.')

## 1. Cargar datos y modelos

In [ ]:
# Datos procesados
df_equipos = leer_csv_s3(BUCKET, 'datos_procesados/historico_equipos_limpio.csv', parse_dates=['Date'])
df_x = leer_csv_s3(BUCKET, 'datos_procesados/X_limpio.csv', parse_dates=['Date'])
df_y = leer_csv_s3(BUCKET, 'datos_procesados/Y_limpio.csv', parse_dates=['Date'])
df_z = leer_csv_s3(BUCKET, 'datos_procesados/Z_limpio.csv', parse_dates=['Date'])

# Modelos
modelo_equipo1 = cargar_pickle_s3(BUCKET, 'modelos/mejor_modelo_equipo1.pkl')
modelo_equipo2 = cargar_pickle_s3(BUCKET, 'modelos/mejor_modelo_equipo2.pkl')
scaler_equipo1 = cargar_pickle_s3(BUCKET, 'modelos/scaler_equipo1.pkl')
scaler_equipo2 = cargar_pickle_s3(BUCKET, 'modelos/scaler_equipo2.pkl')
prophet_equipo1 = cargar_pickle_s3(BUCKET, 'modelos/prophet_equipo1.pkl')
prophet_equipo2 = cargar_pickle_s3(BUCKET, 'modelos/prophet_equipo2.pkl')

# Config de features
obj = s3.get_object(Bucket=BUCKET, Key='modelos/features_config.json')
features_config = json.loads(obj['Body'].read())

print('Datos y modelos cargados.')
print(f'\nRangos de datos:')
print(f'  Equipos: {df_equipos["Date"].min().date()} a {df_equipos["Date"].max().date()}')
print(f'  X:       {df_x["Date"].min().date()} a {df_x["Date"].max().date()}')
print(f'  Y:       {df_y["Date"].min().date()} a {df_y["Date"].max().date()}')
print(f'  Z:       {df_z["Date"].min().date()} a {df_z["Date"].max().date()}')

## 2. Definir horizonte de prediccion

El historico de equipos termina en agosto 2023. X tiene datos hasta abril 2024 (7 meses extra). Y y Z terminan casi al mismo tiempo que el historico.

**Horizonte elegido: 6 meses** (septiembre 2023 a febrero 2024)

**Justificacion:**
- X tiene datos reales hasta abril 2024, lo que permite usar valores reales de X para la proyeccion
- Y y Z deben ser proyectadas con Prophet, y la incertidumbre crece con el horizonte
- 6 meses es un horizonte razonable para planificacion financiera en construccion
- Mas alla de 6 meses la incertidumbre acumulada en Y y Z hace las estimaciones poco confiables

In [ ]:
# Horizonte: 6 meses desde el ultimo dato del historico
fecha_inicio_pred = pd.Timestamp('2023-09-01')
fecha_fin_pred = pd.Timestamp('2024-02-29')
horizonte_meses = 6

print(f'Horizonte de prediccion: {fecha_inicio_pred.date()} a {fecha_fin_pred.date()} ({horizonte_meses} meses)')

## 3. Proyectar materias primas Y y Z con Prophet

X tiene datos reales en el horizonte de prediccion. Y y Z necesitan ser proyectadas.

In [ ]:
# Proyectar Y con Prophet
df_y_prophet = df_y.rename(columns={'Date': 'ds', 'Price': 'y'})

prophet_y = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    interval_width=0.95
)
prophet_y.fit(df_y_prophet)

# Generar fechas futuras (dias habiles)
fechas_futuras_y = pd.bdate_range(start=df_y['Date'].max() + pd.Timedelta(days=1), end=fecha_fin_pred)
future_y = pd.DataFrame({'ds': fechas_futuras_y})
forecast_y = prophet_y.predict(future_y)

print(f'Proyeccion Y: {len(forecast_y)} dias habiles')
print(f'  Rango: {forecast_y["ds"].min().date()} a {forecast_y["ds"].max().date()}')
print(f'  Precio medio proyectado: {forecast_y["yhat"].mean():.2f}')
print(f'  IC 95%: [{forecast_y["yhat_lower"].mean():.2f}, {forecast_y["yhat_upper"].mean():.2f}]')

In [ ]:
# Proyectar Z con Prophet
df_z_prophet = df_z.rename(columns={'Date': 'ds', 'Price': 'y'})

prophet_z = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    interval_width=0.95
)
prophet_z.fit(df_z_prophet)

fechas_futuras_z = pd.bdate_range(start=df_z['Date'].max() + pd.Timedelta(days=1), end=fecha_fin_pred)
future_z = pd.DataFrame({'ds': fechas_futuras_z})
forecast_z = prophet_z.predict(future_z)

print(f'Proyeccion Z: {len(forecast_z)} dias habiles')
print(f'  Rango: {forecast_z["ds"].min().date()} a {forecast_z["ds"].max().date()}')
print(f'  Precio medio proyectado: {forecast_z["yhat"].mean():.2f}')
print(f'  IC 95%: [{forecast_z["yhat_lower"].mean():.2f}, {forecast_z["yhat_upper"].mean():.2f}]')

In [ ]:
# Visualizar proyecciones de Y y Z
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Y
axes[0].plot(df_y['Date'], df_y['Price'], linewidth=0.5, color='steelblue', label='Historico')
axes[0].plot(forecast_y['ds'], forecast_y['yhat'], linewidth=1.5, color='darkorange', label='Proyeccion')
axes[0].fill_between(forecast_y['ds'], forecast_y['yhat_lower'], forecast_y['yhat_upper'],
                      alpha=0.2, color='darkorange', label='IC 95%')
axes[0].axvline(x=df_y['Date'].max(), color='red', linestyle='--', linewidth=1)
axes[0].set_title('Proyeccion Materia Prima Y', fontweight='bold')
axes[0].set_ylabel('Precio')
axes[0].legend()

# Z
axes[1].plot(df_z['Date'], df_z['Price'], linewidth=0.5, color='steelblue', label='Historico')
axes[1].plot(forecast_z['ds'], forecast_z['yhat'], linewidth=1.5, color='darkorange', label='Proyeccion')
axes[1].fill_between(forecast_z['ds'], forecast_z['yhat_lower'], forecast_z['yhat_upper'],
                      alpha=0.2, color='darkorange', label='IC 95%')
axes[1].axvline(x=df_z['Date'].max(), color='red', linestyle='--', linewidth=1)
axes[1].set_title('Proyeccion Materia Prima Z', fontweight='bold')
axes[1].set_ylabel('Precio')
axes[1].set_xlabel('Fecha')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Construir dataset de prediccion

In [ ]:
# Obtener datos reales de X en el horizonte
x_futuro = df_x[(df_x['Date'] >= fecha_inicio_pred) & (df_x['Date'] <= fecha_fin_pred)].copy()
x_futuro = x_futuro.rename(columns={'Price': 'Price_X'})

# Preparar Y proyectada
y_futuro = forecast_y[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
y_futuro = y_futuro.rename(columns={'ds': 'Date', 'yhat': 'Price_Y', 'yhat_lower': 'Price_Y_lower', 'yhat_upper': 'Price_Y_upper'})

# Preparar Z proyectada
z_futuro = forecast_z[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
z_futuro = z_futuro.rename(columns={'ds': 'Date', 'yhat': 'Price_Z', 'yhat_lower': 'Price_Z_lower', 'yhat_upper': 'Price_Z_upper'})

# Merge por fecha
df_pred = x_futuro[['Date', 'Price_X']].merge(y_futuro, on='Date', how='inner')
df_pred = df_pred.merge(z_futuro, on='Date', how='inner')
df_pred = df_pred.sort_values('Date').reset_index(drop=True)

print(f'Dataset de prediccion: {df_pred.shape}')
print(f'Rango: {df_pred["Date"].min().date()} a {df_pred["Date"].max().date()}')
print(f'\nPrimeros registros:')
print(df_pred.head())

## 5. Proyeccion con Prophet (modelo principal para proyeccion)

In [ ]:
# Prophet necesita el formato ds + regressors
proyecciones_prophet = {}

for target, modelo_p in [('Price_Equipo1', prophet_equipo1), ('Price_Equipo2', prophet_equipo2)]:
    df_prophet_pred = df_pred[['Date', 'Price_X', 'Price_Y', 'Price_Z']].copy()
    df_prophet_pred = df_prophet_pred.rename(columns={'Date': 'ds'})
    
    forecast = modelo_p.predict(df_prophet_pred)
    
    proyecciones_prophet[target] = {
        'fecha': df_pred['Date'].values,
        'yhat': forecast['yhat'].values,
        'yhat_lower': forecast['yhat_lower'].values,
        'yhat_upper': forecast['yhat_upper'].values,
        'forecast': forecast
    }
    
    print(f'{target} - Proyeccion Prophet:')
    print(f'  Precio medio proyectado: {forecast["yhat"].mean():.2f}')
    print(f'  IC 95%: [{forecast["yhat_lower"].mean():.2f}, {forecast["yhat_upper"].mean():.2f}]')
    print(f'  Min: {forecast["yhat"].min():.2f}, Max: {forecast["yhat"].max():.2f}')
    print()

## 6. Proyeccion con Lasso (validacion cruzada)

In [ ]:
# Lasso usa solo Price_X, Price_Y, Price_Z (features base dieron el mejor resultado)
proyecciones_lasso = {}

for target, modelo, scaler in [
    ('Price_Equipo1', modelo_equipo1, scaler_equipo1),
    ('Price_Equipo2', modelo_equipo2, scaler_equipo2)
]:
    X_pred = df_pred[['Price_X', 'Price_Y', 'Price_Z']].copy()
    
    # El Lasso R3 usa features_con_interact, necesitamos construirlas
    # Pero los mejores resultados fueron con features base en R1
    # Verificamos que features usa el scaler
    n_features_scaler = scaler.n_features_in_
    
    if n_features_scaler == 3:
        X_scaled = scaler.transform(X_pred)
    else:
        # Si el scaler tiene mas features, usamos solo las 3 base
        # y creamos un scaler nuevo ajustado a los datos historicos
        from sklearn.preprocessing import StandardScaler
        df_hist = leer_csv_s3(BUCKET, 'datos_procesados/historico_equipos_limpio.csv', parse_dates=['Date'])
        scaler_base = StandardScaler()
        scaler_base.fit(df_hist[['Price_X', 'Price_Y', 'Price_Z']])
        X_scaled = scaler_base.transform(X_pred)
        
        # Reentrenar Lasso con features base
        from sklearn.linear_model import Lasso
        modelo_base = Lasso(alpha=0.1)
        y_train = df_hist[target]
        X_train_scaled = scaler_base.transform(df_hist[['Price_X', 'Price_Y', 'Price_Z']])
        modelo_base.fit(X_train_scaled, y_train)
        modelo = modelo_base
    
    y_pred_lasso = modelo.predict(X_scaled)
    
    proyecciones_lasso[target] = {
        'fecha': df_pred['Date'].values,
        'yhat': y_pred_lasso
    }
    
    print(f'{target} - Proyeccion Lasso:')
    print(f'  Precio medio proyectado: {y_pred_lasso.mean():.2f}')
    print(f'  Min: {y_pred_lasso.min():.2f}, Max: {y_pred_lasso.max():.2f}')
    print()

## 7. Visualizacion de proyecciones

In [ ]:
for target in ['Price_Equipo1', 'Price_Equipo2']:
    pp = proyecciones_prophet[target]
    pl = proyecciones_lasso[target]
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 12))
    
    # Panel 1: Historico + proyeccion
    ax = axes[0]
    ax.plot(df_equipos['Date'], df_equipos[target], linewidth=0.6, color='steelblue', label='Historico')
    ax.plot(pp['fecha'], pp['yhat'], linewidth=2, color='darkorange', label='Prophet')
    ax.fill_between(pp['fecha'], pp['yhat_lower'], pp['yhat_upper'],
                     alpha=0.2, color='darkorange', label='IC 95% Prophet')
    ax.plot(pl['fecha'], pl['yhat'], linewidth=2, color='red', linestyle='--', label='Lasso')
    ax.axvline(x=df_equipos['Date'].max(), color='gray', linestyle='--', linewidth=1, label='Inicio proyeccion')
    ax.set_title(f'Proyeccion de Costos - {target}', fontweight='bold', fontsize=14)
    ax.set_ylabel('Precio')
    ax.legend(fontsize=10)
    
    # Panel 2: Zoom en proyeccion
    ax2 = axes[1]
    ax2.plot(pp['fecha'], pp['yhat'], linewidth=2, color='darkorange', label='Prophet', marker='o', markersize=2)
    ax2.fill_between(pp['fecha'], pp['yhat_lower'], pp['yhat_upper'],
                      alpha=0.2, color='darkorange', label='IC 95%')
    ax2.plot(pl['fecha'], pl['yhat'], linewidth=2, color='red', linestyle='--', label='Lasso', marker='s', markersize=2)
    ax2.set_title(f'Zoom Proyeccion - {target}', fontweight='bold', fontsize=14)
    ax2.set_ylabel('Precio')
    ax2.set_xlabel('Fecha')
    ax2.legend(fontsize=10)
    
    plt.tight_layout()
    plt.show()

## 8. Resumen mensual de proyecciones

In [ ]:
resumen_mensual = {}

for target in ['Price_Equipo1', 'Price_Equipo2']:
    pp = proyecciones_prophet[target]
    pl = proyecciones_lasso[target]
    
    df_resumen = pd.DataFrame({
        'Date': pp['fecha'],
        'Prophet': pp['yhat'],
        'Prophet_lower': pp['yhat_lower'],
        'Prophet_upper': pp['yhat_upper'],
        'Lasso': pl['yhat']
    })
    df_resumen['Date'] = pd.to_datetime(df_resumen['Date'])
    df_resumen['Mes'] = df_resumen['Date'].dt.to_period('M')
    
    mensual = df_resumen.groupby('Mes').agg({
        'Prophet': 'mean',
        'Prophet_lower': 'mean',
        'Prophet_upper': 'mean',
        'Lasso': 'mean'
    }).round(2)
    
    print(f'\nRESUMEN MENSUAL - {target}')
    print('=' * 80)
    print(f'{"Mes":12s} {"Prophet":>10s} {"IC Inferior":>12s} {"IC Superior":>12s} {"Ancho IC":>10s} {"Lasso":>10s} {"Diferencia":>10s}')
    print('-' * 80)
    
    meses_dict = {}
    for mes, row in mensual.iterrows():
        ancho_ic = row['Prophet_upper'] - row['Prophet_lower']
        diff = abs(row['Prophet'] - row['Lasso'])
        print(f'{str(mes):12s} {row["Prophet"]:10.2f} {row["Prophet_lower"]:12.2f} {row["Prophet_upper"]:12.2f} {ancho_ic:10.2f} {row["Lasso"]:10.2f} {diff:10.2f}')
        
        meses_dict[str(mes)] = {
            'prophet_mean': row['Prophet'],
            'prophet_lower': row['Prophet_lower'],
            'prophet_upper': row['Prophet_upper'],
            'ancho_ic': round(ancho_ic, 2),
            'lasso_mean': row['Lasso']
        }
    
    resumen_mensual[target] = meses_dict

## 9. Analisis de incertidumbre

In [ ]:
print('ANALISIS DE INCERTIDUMBRE')
print('=' * 60)

incertidumbre = {}

for target in ['Price_Equipo1', 'Price_Equipo2']:
    pp = proyecciones_prophet[target]
    
    ancho_ic = pp['yhat_upper'] - pp['yhat_lower']
    ancho_relativo = ancho_ic / pp['yhat'] * 100
    
    print(f'\n{target}:')
    print(f'  Ancho IC promedio: {ancho_ic.mean():.2f}')
    print(f'  Ancho IC relativo: {ancho_relativo.mean():.1f}%')
    print(f'  Ancho IC al inicio (mes 1): {ancho_ic[:22].mean():.2f}')
    print(f'  Ancho IC al final (mes 6): {ancho_ic[-22:].mean():.2f}')
    print(f'  Crecimiento de incertidumbre: {(ancho_ic[-22:].mean() / ancho_ic[:22].mean() - 1) * 100:.1f}%')
    
    incertidumbre[target] = {
        'ancho_ic_promedio': round(float(ancho_ic.mean()), 2),
        'ancho_ic_relativo_pct': round(float(ancho_relativo.mean()), 1),
        'ancho_ic_mes1': round(float(ancho_ic[:22].mean()), 2),
        'ancho_ic_mes6': round(float(ancho_ic[-22:].mean()), 2)
    }

# Grafica de evolucion de incertidumbre
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for i, target in enumerate(['Price_Equipo1', 'Price_Equipo2']):
    pp = proyecciones_prophet[target]
    ancho = pp['yhat_upper'] - pp['yhat_lower']
    axes[i].plot(pp['fecha'], ancho, color='darkorange', linewidth=1.5)
    axes[i].set_title(f'Evolucion del Ancho del IC - {target}', fontweight='bold')
    axes[i].set_ylabel('Ancho IC 95%')
    axes[i].set_xlabel('Fecha')

plt.tight_layout()
plt.show()

## 10. Guardar proyecciones en S3

In [ ]:
# Guardar dataset de proyecciones
df_proyecciones = df_pred[['Date', 'Price_X', 'Price_Y', 'Price_Z']].copy()

for target in ['Price_Equipo1', 'Price_Equipo2']:
    pp = proyecciones_prophet[target]
    pl = proyecciones_lasso[target]
    nombre = target.replace('Price_', '')
    
    df_proyecciones[f'{nombre}_Prophet'] = pp['yhat']
    df_proyecciones[f'{nombre}_Prophet_lower'] = pp['yhat_lower']
    df_proyecciones[f'{nombre}_Prophet_upper'] = pp['yhat_upper']
    df_proyecciones[f'{nombre}_Lasso'] = pl['yhat']

# Guardar CSV
buffer = StringIO()
df_proyecciones.to_csv(buffer, index=False)
s3.put_object(Bucket=BUCKET, Key='predicciones/proyecciones_equipos.csv', Body=buffer.getvalue())
print('Proyecciones guardadas en S3.')

# Guardar JSON para el agente
proyecciones_agente = {
    'horizonte': {
        'inicio': str(fecha_inicio_pred.date()),
        'fin': str(fecha_fin_pred.date()),
        'meses': horizonte_meses,
        'justificacion': 'X tiene datos reales hasta abr-2024. Y y Z deben proyectarse, y la incertidumbre crece con el horizonte. 6 meses es razonable para planificacion en construccion.'
    },
    'resumen_mensual': resumen_mensual,
    'incertidumbre': incertidumbre,
    'fuentes_datos_proyeccion': {
        'Price_X': 'Datos reales disponibles hasta abr-2024',
        'Price_Y': 'Proyectado con Prophet',
        'Price_Z': 'Proyectado con Prophet'
    },
    'modelos_usados': {
        'principal': 'Prophet (intervalos de confianza nativos)',
        'validacion': 'Lasso (mejor modelo por metricas puntuales)'
    }
}

guardar_json_s3(proyecciones_agente, BUCKET, 'predicciones/proyecciones_resumen.json')

In [ ]:
print('\n' + '=' * 80)
print('RESUMEN DE PROYECCIONES')
print('=' * 80)
print(f'\nHorizonte: {fecha_inicio_pred.date()} a {fecha_fin_pred.date()} ({horizonte_meses} meses)')
print(f'Modelo principal: Prophet (con IC 95%)')
print(f'Modelo de validacion: Lasso')

for target in ['Price_Equipo1', 'Price_Equipo2']:
    pp = proyecciones_prophet[target]
    pl = proyecciones_lasso[target]
    print(f'\n{target}:')
    print(f'  Prophet - Media: {pp["yhat"].mean():.2f}, Rango: [{pp["yhat"].min():.2f}, {pp["yhat"].max():.2f}]')
    print(f'  IC 95%  - Media: [{pp["yhat_lower"].mean():.2f}, {pp["yhat_upper"].mean():.2f}]')
    print(f'  Lasso   - Media: {pl["yhat"].mean():.2f}, Rango: [{pl["yhat"].min():.2f}, {pl["yhat"].max():.2f}]')

print('\nProyecciones guardadas en s3://consulting-dataknow-prueba-tecnica/predicciones/')